# Infoset Parser Unit Tests\n\n**Proper unit tests with assertions and expected values**\n\nThese tests verify correctness, not just that code runs without crashing.

In [ ]:
import sys\nimport os\n\n# Add parent directory to path\ncurrent_dir = os.getcwd()\nif current_dir.endswith('deep_CFR_vNB_integration'):\n    parent_dir = os.path.dirname(current_dir)\nelse:\n    parent_dir = os.path.dirname(os.path.dirname(current_dir))\n\nsys.path.insert(0, parent_dir)\n\nimport torch\nfrom infoset_parser import (\n    parse_infoset_string,\n    parse_infoset_to_network_input,\n    batch_parse_infosets,\n    extract_infoset_features\n)\n\n# Test tracking\ntests_passed = 0\ntests_failed = 0\n\ndef run_test(test_name, test_func):\n    global tests_passed, tests_failed\n    try:\n        test_func()\n        print(f'✓ {test_name} PASSED')\n        tests_passed += 1\n    except AssertionError as e:\n        print(f'✗ {test_name} FAILED: {e}')\n        tests_failed += 1\n    except Exception as e:\n        print(f'✗ {test_name} ERROR: {e}')\n        tests_failed += 1

## 1. Known Input → Expected Output Tests

In [ ]:
def test_simple_preflop_infoset():\n    infoset = 'S0|H:14s0,13s1,10s2|B:|A:'\n    cc, ah = parse_infoset_to_network_input(infoset)\n    assert cc.canonical_hand == ['14s0', '13s1', '10s2']\n    assert cc.canonical_board == []\n    assert ah == ''\n    assert len(cc.canonical_hand) == 3\n\nrun_test('Simple preflop infoset parsing', test_simple_preflop_infoset)

In [ ]:
def test_flop_with_board_and_history():\n    infoset = 'S1|H:14s0,13s0,10s1|B:9s0,8s1,7s2,6s0,5s1|A:CRDD'\n    cc, ah = parse_infoset_to_network_input(infoset)\n    assert cc.canonical_hand == ['14s0', '13s0', '10s1']\n    assert len(cc.canonical_board) == 5\n    assert cc.canonical_board == ['9s0', '8s1', '7s2', '6s0', '5s1']\n    assert ah == 'CRDD'\n    assert len(ah) == 4\n\nrun_test('Flop with board and action history', test_flop_with_board_and_history)

## 2. Round-trip Encoding Tests

In [ ]:
def test_card_encoding_to_tensor():\n    infoset = 'S0|H:14s0,13s1,10s2|B:|A:'\n    cc, ah = parse_infoset_to_network_input(infoset)\n    tensor = cc.get_canonical_hand_tensor()\n    assert tensor[0].item() == 140\n    assert tensor[1].item() == 131\n    assert tensor[2].item() == 102\n    assert tensor.shape == torch.Size([3])\n\nrun_test('Card encoding to tensor values', test_card_encoding_to_tensor)

In [ ]:
def test_board_encoding_to_tensor():\n    infoset = 'S1|H:14s0|B:9s0,8s1,7s2|A:'\n    cc, ah = parse_infoset_to_network_input(infoset)\n    board_tensor = cc.get_canonical_board_tensor()\n    assert board_tensor[0].item() == 90\n    assert board_tensor[1].item() == 81\n    assert board_tensor[2].item() == 72\n    assert board_tensor.shape == torch.Size([3])\n\nrun_test('Board encoding to tensor values', test_board_encoding_to_tensor)

## 3. Action History Character Tests

In [ ]:
def test_all_action_types():\n    test_cases = [\n        ('S0|H:14s0,13s0|B:|A:F', 'F', 'Fold'),\n        ('S0|H:14s0,13s0|B:|A:C', 'C', 'Call'),\n        ('S0|H:14s0,13s0|B:|A:X', 'X', 'Check'),\n        ('S0|H:14s0,13s0|B:|A:D', 'D', 'Discard'),\n        ('S0|H:14s0,13s0|B:|A:r', 'r', 'Small raise'),\n        ('S0|H:14s0,13s0|B:|A:R', 'R', 'Medium raise'),\n        ('S0|H:14s0,13s0|B:|A:B', 'B', 'Big raise'),\n    ]\n    for infoset, expected_ah, desc in test_cases:\n        cc, ah = parse_infoset_to_network_input(infoset)\n        assert ah == expected_ah, f'{desc} failed: expected {expected_ah}, got {ah}'\n\nrun_test('All action type characters', test_all_action_types)

In [ ]:
def test_multi_action_sequence():\n    infoset = 'S1|H:14s0|B:10s0|A:CRBXrDFC'\n    cc, ah = parse_infoset_to_network_input(infoset)\n    assert ah == 'CRBXrDFC'\n    assert len(ah) == 8\n    assert ah[0] == 'C'\n    assert ah[3] == 'X'\n    assert ah[7] == 'C'\n\nrun_test('Multi-action sequence parsing', test_multi_action_sequence)

## 4. Sorting and Canonicalization Tests

In [ ]:
def test_hand_sorting():\n    infoset = 'S0|H:14s0,10s1,5s2|B:|A:'\n    cc, _ = parse_infoset_to_network_input(infoset)\n    ranks = [int(card.split('s')[0]) for card in cc.canonical_hand]\n    assert ranks == sorted(ranks, reverse=True)\n    assert ranks == [14, 10, 5]\n\nrun_test('Hand sorting by rank', test_hand_sorting)

In [ ]:
def test_board_sorting():\n    infoset = 'S1|H:14s0|B:9s0,8s1,7s2,6s0,5s1|A:'\n    cc, _ = parse_infoset_to_network_input(infoset)\n    board_ranks = [int(card.split('s')[0]) for card in cc.canonical_board]\n    assert board_ranks == sorted(board_ranks, reverse=True)\n    assert board_ranks == [9, 8, 7, 6, 5]\n\nrun_test('Board sorting by rank', test_board_sorting)

In [ ]:
def test_same_suit_canonicalization():\n    infoset = 'S0|H:14s0,13s0,10s0|B:|A:'\n    cc, _ = parse_infoset_to_network_input(infoset)\n    suits = [card.split('s')[1] for card in cc.canonical_hand]\n    assert len(set(suits)) == 1\n    assert suits[0] == '0'\n\nrun_test('Same suit canonicalization', test_same_suit_canonicalization)

## 5. Edge Case Tests

In [ ]:
def test_empty_board():\n    infoset = 'S0|H:14s0,13s0,10s0|B:|A:'\n    cc, ah = parse_infoset_to_network_input(infoset)\n    assert cc.canonical_board == []\n    assert len(cc.canonical_board) == 0\n    board_tensor = cc.get_canonical_board_tensor()\n    assert board_tensor.shape == torch.Size([0])\n\nrun_test('Empty board (preflop)', test_empty_board)

In [ ]:
def test_empty_action_history():\n    infoset = 'S0|H:14s0,13s0,10s0|B:|A:'\n    cc, ah = parse_infoset_to_network_input(infoset)\n    assert ah == ''\n    assert len(ah) == 0\n\nrun_test('Empty action history', test_empty_action_history)

In [ ]:
def test_long_action_history():\n    long_history = 'RCXRBCRCXRBCRCXRBCRCXRBC'\n    infoset = f'S2|H:14s0,13s0,10s1|B:9s0,8s1,7s2,6s0,5s1|A:{long_history}'\n    cc, ah = parse_infoset_to_network_input(infoset)\n    assert ah == long_history\n    assert len(ah) == len(long_history), f'Expected {len(long_history)} characters, got {len(ah)}'\n\nrun_test('Long action history preservation', test_long_action_history)

In [ ]:
def test_duplicate_ranks():\n    infoset = 'S1|H:14s0,14s1,10s2|B:13s0,13s1,10s3,9s0,8s1|A:'\n    cc, ah = parse_infoset_to_network_input(infoset)\n    hand_ranks = [int(card.split('s')[0]) for card in cc.canonical_hand]\n    assert hand_ranks.count(14) == 2\n    board_ranks = [int(card.split('s')[0]) for card in cc.canonical_board]\n    assert board_ranks.count(13) == 2\n    assert board_ranks.count(10) == 1\n\nrun_test('Duplicate ranks (pairs)', test_duplicate_ranks)

## 6. Batch Processing Tests

In [ ]:
def test_batch_parsing_consistency():\n    infosets = [\n        'S0|H:14s0,13s0,10s0|B:|A:',\n        'S0|H:12s1,11s2,9s0|B:|A:C',\n        'S1|H:8s0,7s1,6s2|B:5s0,4s1,3s2,2s0,14s1|A:DD',\n    ]\n    cc_list, ah_list = batch_parse_infosets(infosets)\n    for i, infoset in enumerate(infosets):\n        cc_individual, ah_individual = parse_infoset_to_network_input(infoset)\n        assert cc_list[i].canonical_hand == cc_individual.canonical_hand\n        assert cc_list[i].canonical_board == cc_individual.canonical_board\n        assert ah_list[i] == ah_individual\n\nrun_test('Batch parsing consistency', test_batch_parsing_consistency)

In [ ]:
def test_batch_size_preservation():\n    infosets = ['S0|H:14s0|B:|A:', 'S0|H:13s0|B:|A:', 'S0|H:10s0|B:|A:']\n    cc_list, ah_list = batch_parse_infosets(infosets)\n    assert len(cc_list) == len(infosets)\n    assert len(ah_list) == len(infosets)\n    assert len(cc_list) == len(ah_list)\n\nrun_test('Batch size preservation', test_batch_size_preservation)

## 7. Street Number Tests

In [ ]:
def test_street_extraction():\n    test_cases = [('S0|H:14s0|B:|A:', 0), ('S1|H:14s0|B:10s0|A:', 1), ('S2|H:14s0|B:10s0|A:C', 2), ('S3|H:14s0|B:10s0|A:CR', 3)]\n    for infoset, expected_street in test_cases:\n        cc, ah, street = parse_infoset_string(infoset)\n        assert street == expected_street\n\nrun_test('Street number extraction', test_street_extraction)

## Test Summary

In [ ]:
print('\n' + '=' * 70)\nprint('INFOSET PARSER UNIT TEST SUMMARY')\nprint('=' * 70)\nprint(f'\nTests passed: {tests_passed}')\nprint(f'Tests failed: {tests_failed}')\nprint(f'Total tests: {tests_passed + tests_failed}')\nif tests_failed == 0:\n    print('\n✓ ALL TESTS PASSED!')\nelse:\n    print(f'\n✗ {tests_failed} TEST(S) FAILED')\nprint('=' * 70)